# 89 — Bảng lệnh cuối ngày

**Sản phẩm 6.** Chạy một chiến lược danh mục trên dữ liệu mới nhất, lấy trạng
thái danh mục tới phiên đóng cửa gần nhất, rồi xuất **danh sách lệnh cho phiên
kế tiếp** — mã, chiều, khối lượng đã làm tròn lô, giá tham chiếu, trần/sàn — ra
HTML và Excel, kèm một **nhật ký paper-trading** cộng dồn qua các ngày để đối
chiếu lệnh đã đề xuất với giá khớp thật.

> ⚠️ **Notebook này chỉ đọc dữ liệu và ghi file.** Nó không kết nối broker,
> không đặt lệnh thật, và danh sách lệnh là đầu ra của một chiến lược **minh
> hoạ** trên tài khoản giấy — **không phải khuyến nghị đầu tư**.

Ba nguyên tắc quyết định code trông thế nào:

| Nguyên tắc | Vì sao |
|---|---|
| Bảng lệnh do **chính backtest** sinh ra ở thanh cuối | hai bản code — một cho backtest, một cho "live" — sớm muộn sẽ lệch nhau |
| Chỉ dùng phiên **đã đóng cửa** | thanh của phiên đang chạy là một giá đóng cửa chưa tồn tại |
| Giá in trên lệnh là giá **thô** | Sở tính trần/sàn trên giá thô; giá điều chỉnh chỉ tồn tại trong máy |

Mỗi nguyên tắc có một mục đo bằng số ở phần sau: phiên dở dang (mục 8), giá
điều chỉnh (mục 9), tính nhất quán giữa bảng lệnh và backtest (mục 10).

In [1]:
import sys
from pathlib import Path

GOC = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "finlens_examples").is_dir())
sys.path.insert(0, str(GOC))

import datetime as dt
import time

import backtrader as bt
import numpy as np
import pandas as pd
from IPython.display import HTML, display

import finlens
from finlens_examples import (
    ap_dung_theme,
    doc_don_vi,
    duong,
    hom_nay,
    loc_thanh_khoan,
    lui_ngay,
    universe_hose,
)
from finlens_examples import bt_vn
from finlens_examples.charts import GIAM, TANG

ap_dung_theme()
client = finlens.client()
HOM_NAY = hom_nay(client)

THONG_TIN = client.whoami()
GIO_SERVER = dt.datetime.fromisoformat(THONG_TIN["server_time"])
print(f"Giờ server: {GIO_SERVER:%d/%m/%Y %H:%M} (+07) · backtrader {bt.__version__}")

D:\finlens\finlens-python\finlens-python-example\venv\Lib\site-packages\anyio\_core\_tasks.py:275: UpgradeAvailableWarning: finlens 1.5.0 đã có (đang dùng 1.4.0). Nâng cấp: pip install -U finlens
  retval = await self._coro


Giờ server: 20/10/2026 11:21 (+07) · backtrader 1.9.78.123


## 1 · Cấu hình — mọi tham số ở một chỗ

Chiến lược được chọn vì nó **đọc được**, không phải vì nó tốt. Mọi con số ở
đây đều có thể tranh luận; điều không tranh luận là chúng nằm ở một chỗ và bảng
lệnh, backtest, phép kiểm nhất quán đều đọc cùng một bản.

In [2]:
SO_MA_VU_TRU = 30          # rổ thanh khoản cao nhất HOSE — gần với tinh thần VN30
SO_VI_THE = 5              # tối đa 5 mã, mỗi mã ~1/5 giá trị danh mục
NGUONG_GIU = 10            # còn trong top 10 động lượng thì giữ — vùng đệm chống đảo lệnh
PHIEN_DONG_LUONG = 60      # động lượng = lợi suất 60 phiên
PHIEN_MA = 50              # lọc xu hướng: đóng cửa trên SMA50
VON_BAN_DAU = 1_000_000_000  # VND — đơn vị bên trong backtrader là VND
SAN = "HOSE"

THU_MUC_RA = GOC / "output"
THU_MUC_RA.mkdir(exist_ok=True)
NHAT_KY = THU_MUC_RA / "nhat_ky_paper_trading.csv"

## 2 · Vũ trụ: 30 mã HOSE thanh khoản cao nhất

`client.meta` không có danh sách thành phần VN30, nên rổ ở đây là 30 mã HOSE có
giá trị giao dịch bình quân 3 tháng cao nhất — cùng tinh thần "mã lớn, khớp
được khối lượng thật".

Cửa sổ lọc **kết thúc ở hôm qua**, không phải hôm nay: nếu tính cả phiên đang
chạy thì rổ đổi theo giờ bạn bấm chạy, và bảng lệnh 10:00 khác bảng lệnh 14:00
chỉ vì vũ trụ đổi.

⚠️ Chọn rổ bằng thanh khoản *hiện tại* rồi backtest ngược về quá khứ là thiên
lệch sống sót — mã nào mất thanh khoản trong hai năm qua đã bị loại trước khi
backtest bắt đầu. Với bảng lệnh *ngày mai* điều đó không sao; với con số CAGR ở
mục 5 thì có — đọc nó như minh hoạ, không phải bằng chứng.

In [3]:
danh_muc = universe_hose(client)
thanh_khoan = loc_thanh_khoan(
    client,
    danh_muc["symbol"].tolist(),
    start=lui_ngay(HOM_NAY, thang=3),
    end=lui_ngay(HOM_NAY, ngay=1),
)
VU_TRU = thanh_khoan.head(SO_MA_VU_TRU)["symbol"].tolist()

print(f"{len(danh_muc)} cổ phiếu HOSE → {len(thanh_khoan)} mã GTGD ≥ 5 tỷ/phiên → lấy {len(VU_TRU)} mã đầu")
print(", ".join(VU_TRU))

409 cổ phiếu HOSE → 138 mã GTGD ≥ 5 tỷ/phiên → lấy 30 mã đầu
VIC, VHM, SHB, TCB, VIX, FPT, VPB, HPG, HDB, SSI, MSN, MWG, BSR, NVL, PNJ, VCB, MBB, STB, CTG, GEX, VNM, ACB, VCI, MSB, VND, PLX, VJC, BID, PVT, VSC


## 3 · Dữ liệu: hai bản giá, và thanh của phiên đang chạy

Notebook tải **hai** bản giá cho cùng rổ mã:

- **giá điều chỉnh** (mặc định) cho tín hiệu và backtest — lợi suất qua ngày
  chia cổ tức mới liên tục;
- **giá thô** (`adjusted=False`) cho mọi con số in trên lệnh — giá tham chiếu,
  trần, sàn là những thứ Sở công bố trên giá thô.

Đơn vị đọc từ `attrs` **một lần, ngay sau khi tải** rồi giữ trong biến:
`attrs` không sống sót qua `pd.concat` (mục 8 sẽ ghép frame), và
`bt_vn.chuan_bi_frame` từ chối đoán đơn vị khi thiếu nó.

In [4]:
gia_tai = client.eod.stock.ohlcv(VU_TRU, start=lui_ngay(HOM_NAY, nam=2))
gia_tho_tai = client.eod.stock.ohlcv(VU_TRU, start=lui_ngay(HOM_NAY, thang=6), adjusted=False)

DON_VI_GIA = doc_don_vi(gia_tai, "close")
DON_VI_GIA_THO = doc_don_vi(gia_tho_tai, "close")
VND_MOI_DON_VI = {"kVND": 1_000, "VND": 1}[DON_VI_GIA_THO]

print(f"Điều chỉnh: {len(gia_tai):,} dòng · price_basis={gia_tai.attrs['finlens']['price_basis']} · đơn vị {DON_VI_GIA}")
print(f"Thô:        {len(gia_tho_tai):,} dòng · price_basis={gia_tho_tai.attrs['finlens']['price_basis']} · đơn vị {DON_VI_GIA_THO}")

Điều chỉnh: 14,902 dòng · price_basis=adjusted · đơn vị kVND
Thô:        3,690 dòng · price_basis=raw · đơn vị kVND


In [5]:
gia = bt_vn.bo_phien_chua_dong(gia_tai, client)
gia_tho = bt_vn.bo_phien_chua_dong(gia_tho_tai, client)

T = gia["date"].max()  # phiên đã đóng cửa gần nhất — "hôm nay" của bảng lệnh
CUOI_TAI = gia_tai["date"].max()
PHIEN_DO_DANG = CUOI_TAI if CUOI_TAI > T else None

print(f"Phiên đóng cửa gần nhất T = {T:%d/%m/%Y}")
if PHIEN_DO_DANG is not None:
    dd = gia_tai[gia_tai["date"] == PHIEN_DO_DANG]
    kl_bq = gia[gia["date"] > T - pd.Timedelta(days=30)].groupby("symbol", observed=True)["volume"].mean()
    ty_le = (dd.set_index("symbol")["volume"] / kl_bq).median()
    print(
        f"Dữ liệu tải về có thanh {PHIEN_DO_DANG:%d/%m} của phiên ĐANG CHẠY ({len(dd)} mã) — đã bỏ. "
        f"Khối lượng của nó mới bằng {ty_le:.0%} bình quân (trung vị theo mã) lúc {GIO_SERVER:%H:%M}."
    )
else:
    print("Không có thanh dở dang: lần chạy này sau giờ đóng cửa hoặc ngoài ngày giao dịch.")

Phiên đóng cửa gần nhất T = 19/10/2026
Dữ liệu tải về có thanh 20/10 của phiên ĐANG CHẠY (30 mã) — đã bỏ. Khối lượng của nó mới bằng 31% bình quân (trung vị theo mã) lúc 11:21.


## 4 · Chiến lược — viết ra để tranh luận được

| Bước | Luật |
|---|---|
| Xếp hạng | 30 mã theo động lượng 60 phiên (`bt.ind.RateOfChange`) |
| Bán | đang giữ **và** (đóng cửa dưới SMA50 **hoặc** rơi khỏi top 10) — chỉ bán phần đã về tài khoản (T+2) |
| Mua | mã xếp đầu chưa giữ, động lượng > 0 và trên SMA50, cho tới khi đủ 5 vị thế |
| Khối lượng | 1/5 giá trị danh mục × 0,98, không vượt tiền mặt, làm tròn **xuống** lô 100 bằng `bt_vn.lam_tron_lo` |
| Khớp | lệnh thị trường khớp **giá mở cửa phiên sau** — ngoài đời là lệnh ATO; không khớp được thì huỷ, như lệnh ngày |
| Luật Việt Nam | `PhiCoPhieuVN` (0,15% hai chiều + 0,1% thuế bán) · `KhopTheoBienDo` (không khớp ở phiên trần/sàn trắng bên) · `ChienLuocVN` (T+2) |

`bt_vn.LoVN` không dùng ở đây: nó mua bằng 95% tiền mặt *hiện có* — đúng cho một
mã, sai cho danh mục 5 vị thế, vì lệnh mua đầu tiên sẽ ăn hết tiền của bốn lệnh
sau. Khối lượng tính thẳng bằng `lam_tron_lo`, là chính hàm `LoVN` dùng bên
trong.

Mỗi lệnh chiến lược tạo ra được **ghi kèm lý do** vào `so_lenh`. Bảng lệnh của
phiên mai không phải một hàm riêng: nó là các dòng `so_lenh` của thanh cuối cùng.

Tham số `nhin_truoc` chỉ để chứng minh cạm bẫy ở mục 10 — bảng lệnh thật luôn
chạy với `False`.

In [6]:
class DaTangVN(bt_vn.ChienLuocVN):
    """Đà tăng trên rổ thanh khoản cao, tối đa K vị thế, đủ luật thị trường Việt Nam."""

    params = (
        ("k", SO_VI_THE),
        ("nguong_giu", NGUONG_GIU),
        ("n_dl", PHIEN_DONG_LUONG),
        ("n_ma", PHIEN_MA),
        ("nhin_truoc", False),  # ⚠️ chỉ dùng ở mục 10
    )

    def __init__(self):
        super().__init__()  # không gọi thì sổ T+2 rỗng và ban_duoc() luôn trả 0
        self.dl = {d: bt.ind.RateOfChange(d.close, period=self.p.n_dl) for d in self.datas}
        self.ma = {d: bt.ind.SMA(d.close, period=self.p.n_ma) for d in self.datas}
        self.so_lenh = []
        self.ngay_mua = {}
        self.dang_cho = []
        self.dem = {"t2_chan": 0, "bi_tu_choi": 0, "huy_khong_khop": 0}

    def notify_order(self, order):
        super().notify_order(order)
        if order.status == order.Completed and order.isbuy():
            self.ngay_mua[order.data._name] = pd.Timestamp(order.data.datetime.date(0))
        elif order.status in (order.Margin, order.Rejected):
            self.dem["bi_tu_choi"] += 1

    def prenext(self):
        # Mã niêm yết muộn chưa đủ thanh cho chỉ báo: vẫn chạy, next() tự bỏ qua mã đó
        self.next()

    def _gia_tinh_khoi_luong(self, d):
        if not self.p.nhin_truoc:
            return d.close[0]
        try:
            return d.open[1]  # preload=True: [1] đọc được giá mở cửa của NGÀY MAI
        except IndexError:  # ở thanh cuối không có ngày mai → lùi về giá đóng cửa
            return d.close[0]

    def _ghi(self, ngay, d, chieu, kl, ly_do, hang):
        self.so_lenh.append(
            {"ngay": pd.Timestamp(ngay), "ma": d._name, "chieu": chieu, "khoi_luong": int(kl),
             "dong_luong_pct": round(self.dl[d][0] * 100, 2), "hang": hang, "ly_do": ly_do}
        )

    def next(self):
        for o in self.dang_cho:  # lệnh ngày: phiên trước không khớp được thì huỷ
            if o.alive():
                self.cancel(o)
                self.dem["huy_khong_khop"] += 1
        self.dang_cho = []

        ngay = max(d.datetime.date(0) for d in self.datas if len(d))
        du_thanh = max(self.p.n_dl, self.p.n_ma)
        co_mat = [d for d in self.datas if len(d) > du_thanh and d.datetime.date(0) == ngay]
        if not co_mat:
            return
        xep = sorted(co_mat, key=lambda d: self.dl[d][0], reverse=True)
        hang = {d: i + 1 for i, d in enumerate(xep)}
        n = len(xep)

        # 1) Bán trước — tiền bán về là một phần sức mua của lệnh mua cùng phiên
        con_giu, tien_ban = [], 0.0
        for d in self.datas:
            if self.getposition(d).size <= 0:
                continue
            ly_do = None
            if d in hang:
                if d.close[0] < self.ma[d][0]:
                    ly_do = f"đóng cửa dưới SMA{self.p.n_ma}"
                elif hang[d] > self.p.nguong_giu:
                    ly_do = f"rơi khỏi top {self.p.nguong_giu} động lượng (hạng {hang[d]}/{n})"
            if ly_do is None:
                con_giu.append(d)
                continue
            o = self.ban(d)  # chỉ bán phần đã về — T+2
            if o is None:
                self.dem["t2_chan"] += 1
                con_giu.append(d)
                continue
            self.dang_cho.append(o)
            tien_ban += abs(o.size) * d.close[0]
            self._ghi(ngay, d, "BÁN", abs(o.size), ly_do, hang[d])

        # 2) Mua cho đủ chỗ trống
        cho_trong = self.p.k - len(con_giu)
        if cho_trong <= 0:
            return
        muc_tieu = self.broker.getvalue() / self.p.k * 0.98
        tien = self.broker.getcash() + tien_ban * (1 - 0.0025)  # trừ phí + thuế bán
        for d in xep:
            if cho_trong <= 0:
                break
            if self.getposition(d).size > 0 or self.dl[d][0] <= 0 or d.close[0] <= self.ma[d][0]:
                continue
            gia_ = self._gia_tinh_khoi_luong(d)
            kl = bt_vn.lam_tron_lo(min(muc_tieu, tien * 0.98) / gia_)
            if kl <= 0:
                break
            self.dang_cho.append(self.buy(data=d, size=kl))
            tien -= kl * gia_ * 1.0015
            cho_trong -= 1
            self._ghi(
                ngay, d, "MUA", kl,
                f"động lượng {self.p.n_dl} phiên {self.dl[d][0] * 100:+.1f}%, hạng {hang[d]}/{n}, trên SMA{self.p.n_ma}",
                hang[d],
            )

In [7]:
COT_LENH = ["ngay", "ma", "chieu", "khoi_luong", "dong_luong_pct", "hang", "ly_do"]


def chay(frame: pd.DataFrame, *, nhin_truoc: bool = False) -> DaTangVN:
    """Một lần backtest trên ``frame`` (dạng long, nhiều mã). Trả về strategy đã chạy."""
    cerebro = bt.Cerebro(stdstats=False)
    for ma in VU_TRU:
        if (frame["symbol"] == ma).any():
            # don_vi_gia truyền tường minh: frame ở mục 8 là kết quả pd.concat, đã mất attrs
            cerebro.adddata(bt_vn.feed_tu_finlens(frame, ma=ma, don_vi_gia=DON_VI_GIA))
    cerebro.broker.setcash(VON_BAN_DAU)
    cerebro.broker.addcommissioninfo(bt_vn.PhiCoPhieuVN())
    khop = bt_vn.KhopTheoBienDo(SAN)
    cerebro.broker.set_filler(khop)
    cerebro.addstrategy(DaTangVN, nhin_truoc=nhin_truoc)
    bt_vn.them_analyzer(cerebro)
    strat = cerebro.run()[0]
    strat.dem["tran_san_chan"] = khop.so_lan_chan
    return strat


def lenh_cua(strat: DaTangVN, ngay) -> pd.DataFrame:
    """Các lệnh strategy đã TẠO ở thanh ``ngay`` (sẽ khớp ở phiên sau đó)."""
    l = pd.DataFrame(strat.so_lenh, columns=COT_LENH)
    return l[l["ngay"] == pd.Timestamp(ngay)].reset_index(drop=True)

## 5 · Backtest lịch sử — để có trạng thái danh mục tới phiên T

Mục đích của lần chạy này là **trạng thái**: đang giữ mã nào, bao nhiêu cổ
phiếu, phần nào đã về tài khoản, tiền mặt còn bao nhiêu. Thành tích lịch sử in
ra để biết chiến lược không làm gì kỳ quặc, không phải để chứng minh nó tốt.

In [8]:
t0 = time.perf_counter()
s = chay(gia)
kq = bt_vn.ket_qua(s)
tt = kq["tom_tat"]
print(f"Backtest {len(s.datas)} mã · {kq['loi_suat'].index.min():%d/%m/%Y} → {kq['loi_suat'].index.max():%d/%m/%Y} "
      f"· {time.perf_counter() - t0:.1f} giây")
print(f"  CAGR {tt['cagr']:+.2%} · MDD {tt['mdd']:.2%} · Sharpe (rf=0) {tt['sharpe']:.2f}")
print(f"  {tt['so_lenh_khop']} lần khớp · {tt['so_vong_dong']} vòng đã đóng · tỷ lệ thắng {tt['ty_le_thang']:.0%}")
print(f"  Luật VN đã chặn: {s.dem['t2_chan']} lần muốn bán khi hàng chưa về (T+2) · "
      f"{s.dem['tran_san_chan']} lần khớp ở phiên trần/sàn trắng bên · {s.dem['bi_tu_choi']} lệnh bị broker từ chối")

Backtest 30 mã · 21/10/2024 → 19/10/2026 · 3.0 giây
  CAGR +33.51% · MDD -31.05% · Sharpe (rf=0) 1.21
  209 lần khớp · 102 vòng đã đóng · tỷ lệ thắng 33%
  Luật VN đã chặn: 14 lần muốn bán khi hàng chưa về (T+2) · 0 lần khớp ở phiên trần/sàn trắng bên · 6 lệnh bị broker từ chối


In [9]:
vni = client.eod.index.ohlcv("VNINDEX", start=lui_ngay(HOM_NAY, nam=2))
vni = vni[vni["date"] <= T].set_index("date")["close"]

von = kq["duong_von"]
so_sanh = pd.concat(
    {
        "Chiến lược (sau phí, thuế)": von / von.iloc[0] * 100,
        "VNINDEX": vni.reindex(von.index).ffill() / vni.reindex(von.index).ffill().iloc[0] * 100,
    },
    axis=1,
).reset_index().melt(id_vars="date", var_name="chuỗi", value_name="giá trị")

duong(
    so_sanh, x="date", y="giá trị", theo="chuỗi",
    tieu_de="Đường vốn so với VNINDEX — gốc 100",
    phu_de=f"{len(VU_TRU)} mã HOSE thanh khoản cao · tối đa {SO_VI_THE} vị thế · tới phiên {T:%d/%m/%Y}",
    nhan_y="chỉ số (gốc 100)",
)

## 6 · Trạng thái danh mục ở phiên T

Cột **bán được phiên tới** là `s.ban_duoc(d)` của sổ T+2, gọi sau khi
`cerebro.run()` đã xong: hàng mua khớp hôm qua chưa về thì có tín hiệu bán cũng
chưa bán được, và phần đã nằm trong một lệnh bán đang chờ khớp không được tính
lại lần nữa.

In [10]:
dong = []
for d in s.datas:
    vt = s.getposition(d)
    if vt.size <= 0:
        continue
    dong.append({
        "ma": d._name,
        "khoi_luong": int(vt.size),
        "gia_von": vt.price,
        "dong_cua_T": d.close[0],
        "gia_tri": vt.size * d.close[0],
        "lai_lo_pct": (d.close[0] / vt.price - 1) * 100,
        "ngay_mua": s.ngay_mua.get(d._name),
        "ban_duoc_phien_toi": s.ban_duoc(d),
    })
danh_muc_T = pd.DataFrame(dong)
GIA_TRI_DM = s.broker.getvalue()
TIEN_MAT = s.broker.getcash()

print(f"Giá trị danh mục giấy: {GIA_TRI_DM:,.0f} VND · tiền mặt {TIEN_MAT:,.0f} VND ({TIEN_MAT / GIA_TRI_DM:.1%}) "
      f"· {len(danh_muc_T)} vị thế")
danh_muc_T.style.format(
    {"khoi_luong": "{:,.0f}", "gia_von": "{:,.0f}", "dong_cua_T": "{:,.0f}", "gia_tri": "{:,.0f}",
     "lai_lo_pct": "{:+.2f}%", "ban_duoc_phien_toi": "{:,.0f}", "ngay_mua": "{:%d/%m/%Y}"}, na_rep="—"
).hide(axis="index")

Giá trị danh mục giấy: 1,766,290,108 VND · tiền mặt 9,270,108 VND (0.5%) · 5 vị thế


ma,khoi_luong,gia_von,dong_cua_T,gia_tri,lai_lo_pct,ngay_mua,ban_duoc_phien_toi
TCB,"10,300","32,900","33,250","342,475,000",+1.06%,16/10/2026,"10,300"
VPB,"11,900","23,800","24,000","285,600,000",+0.84%,12/10/2026,"11,900"
HDB,"16,500","20,580","22,750","375,375,000",+10.54%,14/08/2026,"16,500"
MSB,"24,700","14,300","14,800","365,560,000",+3.50%,01/10/2026,"24,700"
PVT,"16,100","22,050","24,100","388,010,000",+9.30%,11/09/2026,"16,100"


## 7 · Bảng lệnh cho phiên kế tiếp

Khối lượng và lý do lấy thẳng từ `so_lenh` của thanh T. Mọi giá in ra là
**giá thô** (`adjusted=False`) đổi sang VND:

- **giá tham chiếu** của phiên mai = giá đóng cửa thô phiên T (HOSE);
- **trần/sàn** = `bt_vn.tran_san(tham_chiếu, "HOSE")` — ±7%, trần làm tròn
  xuống, sàn làm tròn lên theo **bước giá** `bt_vn.buoc_gia` (10 / 50 / 100 đồng);
- **giá trị lệnh** tính theo giá tham chiếu — giá khớp ATO thật sẽ khác.

Loại lệnh là **ATO** vì backtest khớp ở giá mở cửa; đặt lệnh LO ở giá khác là
đang giao dịch một chiến lược khác với chiến lược đã backtest.

In [11]:
def lap_bang_lenh(lenh: pd.DataFrame, gia_tho: pd.DataFrame) -> pd.DataFrame:
    """Gắn giá tham chiếu / trần / sàn (VND, giá thô) vào các lệnh ``so_lenh``.

    Tham chiếu của phiên sau = đóng cửa thô của chính phiên tín hiệu.
    """
    if lenh.empty:
        return pd.DataFrame(columns=["ngay_tin_hieu", "ma", "chieu", "khoi_luong", "loai_lenh", "gia_tham_chieu",
                                     "gia_tran", "gia_san", "buoc_gia", "gia_tri_lenh", "ly_do"])
    dong_cua = gia_tho.set_index(["symbol", "date"])["close"]
    ra = []
    for _, l in lenh.iterrows():
        tc = float(dong_cua.loc[(l["ma"], l["ngay"])]) * VND_MOI_DON_VI
        tran, san = bt_vn.tran_san(tc, SAN)
        ra.append({
            "ngay_tin_hieu": l["ngay"], "ma": l["ma"], "chieu": l["chieu"], "khoi_luong": int(l["khoi_luong"]),
            "loai_lenh": "ATO", "gia_tham_chieu": tc, "gia_tran": tran, "gia_san": san,
            "buoc_gia": bt_vn.buoc_gia(tc, SAN), "gia_tri_lenh": l["khoi_luong"] * tc, "ly_do": l["ly_do"],
        })
    return pd.DataFrame(ra)


bang_lenh = lap_bang_lenh(lenh_cua(s, T), gia_tho)

print(f"Bảng lệnh lập sau phiên {T:%d/%m/%Y} — áp dụng cho phiên giao dịch kế tiếp")
if PHIEN_DO_DANG is not None:
    print(f"⚠️ Lần chạy này diễn ra lúc {GIO_SERVER:%H:%M} ngày {GIO_SERVER:%d/%m}, khi phiên kế tiếp đó ĐÃ mở cửa: "
          "lệnh ATO đã lỡ. Chạy sau 15:00 để có bảng lệnh đúng lúc.")
if bang_lenh.empty:
    print("Không có lệnh: không vị thế nào chạm luật bán, và không có chỗ trống hoặc ứng viên đạt luật mua.")
else:
    print(f"{len(bang_lenh)} lệnh · mua {bang_lenh.loc[bang_lenh['chieu'] == 'MUA', 'gia_tri_lenh'].sum():,.0f} VND "
          f"· bán {bang_lenh.loc[bang_lenh['chieu'] == 'BÁN', 'gia_tri_lenh'].sum():,.0f} VND")

Bảng lệnh lập sau phiên 19/10/2026 — áp dụng cho phiên giao dịch kế tiếp
⚠️ Lần chạy này diễn ra lúc 11:21 ngày 20/10, khi phiên kế tiếp đó ĐÃ mở cửa: lệnh ATO đã lỡ. Chạy sau 15:00 để có bảng lệnh đúng lúc.
Không có lệnh: không vị thế nào chạm luật bán, và không có chỗ trống hoặc ứng viên đạt luật mua.


In [12]:
TEN_COT = {
    "ma": "Mã", "chieu": "Chiều", "khoi_luong": "Khối lượng", "loai_lenh": "Loại lệnh",
    "gia_tham_chieu": "Giá tham chiếu", "gia_tran": "Giá trần", "gia_san": "Giá sàn", "buoc_gia": "Bước giá",
    "gia_tri_lenh": "Giá trị lệnh (VND)", "ly_do": "Lý do",
}


def hien_bang_lenh(bang: pd.DataFrame):
    """Styler cho bảng lệnh — dùng cho cả notebook lẫn file HTML."""
    hien = bang.drop(columns="ngay_tin_hieu").rename(columns=TEN_COT)
    so = ["Khối lượng", "Giá tham chiếu", "Giá trần", "Giá sàn", "Bước giá", "Giá trị lệnh (VND)"]
    return (
        hien.style.format({c: "{:,.0f}" for c in so})
        .map(lambda v: f"color:{TANG};font-weight:600" if v == "MUA" else f"color:{GIAM};font-weight:600",
             subset=["Chiều"])
        .hide(axis="index")
    )


hien_bang_lenh(bang_lenh)

Mã,Chiều,Khối lượng,Loại lệnh,Giá tham chiếu,Giá trần,Giá sàn,Bước giá,Giá trị lệnh (VND),Lý do


## 8 · Cạm bẫy (a): chạy giữa phiên

`bo_phien_chua_dong` bỏ thanh của phiên đang chạy dựa trên đồng hồ server. Bỏ
qua bước đó thì thanh dở dang trở thành "phiên T": chỉ báo tính trên một giá
đóng cửa chưa tồn tại, và bảng lệnh in ra là bảng lệnh cho *ngày kia*.

Trước hết, lần chạy này — nếu nó diễn ra trong giờ giao dịch:

In [13]:
if PHIEN_DO_DANG is not None:
    s_dd = chay(gia_tai)  # không bỏ thanh dở dang
    bang_dd = lap_bang_lenh(lenh_cua(s_dd, PHIEN_DO_DANG), gia_tho_tai)
    print(f"Đúng  (T = {T:%d/%m}, đã đóng):        {len(bang_lenh)} lệnh "
          + ", ".join(f"{r.chieu} {r.ma} {r.khoi_luong:,}" for r in bang_lenh.itertuples()))
    print(f"Sai   (T = {PHIEN_DO_DANG:%d/%m} lúc {GIO_SERVER:%H:%M}): {len(bang_dd)} lệnh "
          + ", ".join(f"{r.chieu} {r.ma} {r.khoi_luong:,}" for r in bang_dd.itertuples()))
    print(f"Giá trị danh mục: {s.broker.getvalue():,.0f} → {s_dd.broker.getvalue():,.0f} VND "
          f"({s_dd.broker.getvalue() / s.broker.getvalue() - 1:+.2%}) — danh mục được 'định giá' bằng giá chưa đóng cửa")
else:
    print("Lần chạy này không có thanh dở dang — phần dựng lại bên dưới vẫn đo được cạm bẫy.")

Đúng  (T = 19/10, đã đóng):        0 lệnh 
Sai   (T = 20/10 lúc 11:21): 0 lệnh 
Giá trị danh mục: 1,766,290,108 → 1,757,175,108 VND (-0.52%) — danh mục được 'định giá' bằng giá chưa đóng cửa


Một lần chạy là một mẫu. Để đo cho có số, dựng lại **thanh dở dang của 10 phiên
đã qua** từ dữ liệu intraday 5 phút — mỗi phiên cắt ở **11:30, giờ nghỉ trưa**,
lúc rất dễ mở notebook ra chạy — rồi so bảng lệnh tính trên thanh đó với bảng
lệnh tính trên thanh đóng cửa thật.

Ba chi tiết kỹ thuật:

1. Intraday tải theo cửa sổ `max_intraday_days − 1` ngày lịch, vì cả hai đầu
   `start`/`end` đều được tính.
2. Intraday là **giá thô**; thanh dựng lại được nhân với hệ số điều chỉnh của
   chính phiên đó (`đóng cửa điều chỉnh ÷ đóng cửa thô`) để nối liền với lịch
   sử điều chỉnh.
3. Frame sau `pd.concat` mất `attrs` — `chay()` vẫn chạy được chỉ vì nó luôn
   truyền `don_vi_gia=` cho `feed_tu_finlens`.

In [14]:
SO_PHIEN_DUNG_LAI = 10
MOC_CHUP = dt.time(11, 30)

phien_dong = list(pd.DatetimeIndex(gia["date"].unique()).sort_values())
phien_dung_lai = phien_dong[-SO_PHIEN_DUNG_LAI:]
max_ngay = THONG_TIN["limits"]["max_intraday_days"]

cac_khung, bat_dau = [], phien_dung_lai[0].date()
while bat_dau <= T.date():
    ket_thuc = min(bat_dau + dt.timedelta(days=max_ngay - 1), T.date())
    cac_khung.append(client.intraday.stock.ohlcv(VU_TRU, start=bat_dau.isoformat(), end=ket_thuc.isoformat(),
                                                 interval="5min"))
    bat_dau = ket_thuc + dt.timedelta(days=1)
assert doc_don_vi(cac_khung[0], "close") == DON_VI_GIA_THO and cac_khung[0].attrs["finlens"]["price_basis"] == "raw"
intra = pd.concat(cac_khung, ignore_index=True)

intra["date"] = intra["time"].dt.tz_localize(None).dt.normalize()
thanh_trua = (
    intra[intra["time"].dt.time < MOC_CHUP]
    .groupby(["symbol", "date"], observed=True)
    .agg(open=("open", "first"), high=("high", "max"), low=("low", "min"), close=("close", "last"),
         volume=("volume", "sum"))
    .reset_index()
)
# Hệ số điều chỉnh của từng (mã, phiên): đưa thanh thô về cùng gốc với lịch sử điều chỉnh
he_so = (gia.set_index(["symbol", "date"])["close"] / gia_tho.set_index(["symbol", "date"])["close"]).rename("he_so")
thanh_trua = thanh_trua.join(he_so, on=["symbol", "date"])
thanh_trua[["open", "high", "low", "close"]] = thanh_trua[["open", "high", "low", "close"]].mul(thanh_trua["he_so"], axis=0)

ty_le_kl = (thanh_trua.set_index(["symbol", "date"])["volume"] / gia_tho.set_index(["symbol", "date"])["volume"]).dropna()
print(f"{len(intra):,} thanh 5 phút → {len(thanh_trua)} thanh 'lúc 11:30' trên {thanh_trua['date'].nunique()} phiên "
      f"({len(cac_khung)} lần gọi intraday)")
print(f"Thanh lúc 11:30 chứa trung vị {ty_le_kl.median():.0%} khối lượng cả phiên")

13,861 thanh 5 phút → 300 thanh 'lúc 11:30' trên 10 phiên (2 lần gọi intraday)
Thanh lúc 11:30 chứa trung vị 44% khối lượng cả phiên


In [15]:
def doi_chieu(dung: pd.DataFrame, thu: pd.DataFrame) -> pd.DataFrame:
    """So hai danh sách lệnh theo (mã, chiều); phân loại từng dòng."""
    a = dung[["ma", "chieu", "khoi_luong"]]
    b = thu[["ma", "chieu", "khoi_luong"]]
    m = a.merge(b, on=["ma", "chieu"], how="outer", suffixes=("_dung", "_thu"))
    m["ket_qua"] = np.select(
        [m["khoi_luong_thu"].isna(), m["khoi_luong_dung"].isna(), m["khoi_luong_dung"] == m["khoi_luong_thu"]],
        ["thiếu lệnh", "lệnh thừa", "trùng"],
        default="lệch khối lượng",
    )
    return m


t0 = time.perf_counter()
dong = []
for d in phien_dung_lai:
    frame = pd.concat([gia[gia["date"] < d], thanh_trua.loc[thanh_trua["date"] == d, gia.columns]], ignore_index=True)
    st = chay(frame)
    m = doi_chieu(lenh_cua(s, d), lenh_cua(st, d))
    dong.append({
        "phiên": d, "lệnh khi đủ phiên": int(m["khoi_luong_dung"].notna().sum()),
        "lệnh lúc 11:30": int(m["khoi_luong_thu"].notna().sum()), "trùng": int((m["ket_qua"] == "trùng").sum()),
        "khác": "; ".join(
            f"{r.chieu} {r.ma}: {r.khoi_luong_dung:,.0f} → {r.khoi_luong_thu:,.0f}".replace("nan", "—")
            for r in m[m["ket_qua"] != "trùng"].itertuples()
        ),
    })
dung_lai = pd.DataFrame(dong)

co_lenh = dung_lai[(dung_lai["lệnh khi đủ phiên"] > 0) | (dung_lai["lệnh lúc 11:30"] > 0)]
sai_phien = (dung_lai["khác"] != "").sum()
print(f"{len(phien_dung_lai)} phiên dựng lại · {time.perf_counter() - t0:.0f} giây")
print(f"Bảng lệnh lúc 11:30 khác bảng lệnh đúng ở {sai_phien}/{len(co_lenh)} phiên có lệnh · "
      f"{dung_lai['trùng'].sum()}/{dung_lai['lệnh khi đủ phiên'].sum()} lệnh đúng được giữ nguyên")
dung_lai.style.format({"phiên": "{:%d/%m/%Y}"}).hide(axis="index")

10 phiên dựng lại · 31 giây
Bảng lệnh lúc 11:30 khác bảng lệnh đúng ở 2/2 phiên có lệnh · 0/4 lệnh đúng được giữ nguyên


phiên,lệnh khi đủ phiên,lệnh lúc 11:30,trùng,khác
06/10/2026,0,0,0,
07/10/2026,0,0,0,
08/10/2026,0,0,0,
09/10/2026,2,0,0,"BÁN VIC: 1,200 → —; MUA VPB: 11,900 → —"
12/10/2026,0,0,0,
13/10/2026,0,0,0,
14/10/2026,0,0,0,
15/10/2026,2,0,0,"BÁN BSR: 12,100 → —; MUA TCB: 10,300 → —"
16/10/2026,0,0,0,
19/10/2026,0,0,0,


In [16]:
# Vì sao bảng lệnh lúc 11:30 khác: xét lại luật bán "đóng cửa dưới SMA50" trên hai giá
dong = []
for d in phien_dung_lai:
    for l in lenh_cua(s, d).itertuples():
        if l.chieu != "BÁN" or not l.ly_do.startswith("đóng cửa dưới"):
            continue
        dc_ma = gia.loc[(gia["symbol"] == l.ma) & (gia["date"] <= d), "close"] * VND_MOI_DON_VI
        trua = float(thanh_trua.loc[(thanh_trua["symbol"] == l.ma) & (thanh_trua["date"] == d), "close"].iloc[0]) * VND_MOI_DON_VI
        dong.append({"phiên": d, "mã": l.ma, "đóng cửa": dc_ma.iloc[-1], "SMA50 (đóng cửa)": dc_ma.iloc[-PHIEN_MA:].mean(),
                     "giá 11:30": trua, "SMA50 (lúc 11:30)": (dc_ma.iloc[-PHIEN_MA:-1].sum() + trua) / PHIEN_MA})
pd.DataFrame(dong).style.format({"phiên": "{:%d/%m/%Y}", **{c: "{:,.0f}" for c in
    ["đóng cửa", "SMA50 (đóng cửa)", "giá 11:30", "SMA50 (lúc 11:30)"]}}).hide(axis="index")

phiên,mã,đóng cửa,SMA50 (đóng cửa),giá 11:30,SMA50 (lúc 11:30)
09/10/2026,VIC,"225,500","226,388","228,000","226,438"
15/10/2026,BSR,"28,200","28,533","28,750","28,544"


Trong 10 phiên dựng lại có 2 phiên có lệnh — 09/10 và 15/10 — và bảng lệnh lúc
11:30 **sai ở cả hai**: 4 lệnh đúng thì không lệnh nào xuất hiện. Bảng trên cho
thấy cơ chế: lúc nghỉ trưa VIC và BSR còn đứng **trên** SMA50, chỉ thủng xuống
trong phiên chiều. Không có lệnh bán thì không có chỗ trống, nên lệnh mua đi kèm
(VPB, TCB) cũng biến mất.

Điều đáng sợ là cách nó sai: notebook không báo lỗi gì, nó in ra một bảng lệnh
hoàn toàn hợp lệ — chỉ là rỗng ở đúng hai ngày cần hành động. Và khi bảng lệnh
lúc trưa *có* lệnh, khối lượng của nó tính trên một giá đóng cửa chưa tồn tại,
trên một danh mục được định giá bằng giá giữa phiên.

## 9 · Cạm bẫy (b): trần/sàn tính trên giá điều chỉnh

Giá điều chỉnh của finlens là **điều chỉnh lùi**: phiên mới nhất giữ nguyên giá
thô, mọi phiên *trước* ngày GDKHQ bị nhân một hệ số. Hệ quả kỳ lạ là bảng lệnh
của hôm nay gần như luôn đúng dù bạn dùng nhầm giá — lỗi chỉ lộ ra khi bạn
**dựng lại bảng lệnh của một ngày đã qua** (để đối chiếu nhật ký, để kiểm
backtest), sau khi mã đó GDKHQ.

Phép đo: với mọi lệnh backtest đã đặt trong 6 tháng qua — đúng những bảng lệnh
notebook này sẽ in nếu chạy mỗi ngày — tính trần/sàn hai cách rồi so.

In [17]:
dc = gia.set_index(["symbol", "date"])["close"] * VND_MOI_DON_VI
tho = gia_tho.set_index(["symbol", "date"])["close"] * VND_MOI_DON_VI

lich_su = pd.DataFrame(s.so_lenh, columns=COT_LENH)
lich_su = lich_su[lich_su["ngay"] >= gia_tho["date"].min()].reset_index(drop=True)

dong = []
for l in lich_su.itertuples():
    tc_dc, tc_tho = float(dc.loc[(l.ma, l.ngay)]), float(tho.loc[(l.ma, l.ngay)])
    (tran_dc, san_dc), (tran_tho, san_tho) = bt_vn.tran_san(tc_dc, SAN), bt_vn.tran_san(tc_tho, SAN)
    dong.append({"phiên tín hiệu": l.ngay, "mã": l.ma, "chiều": l.chieu,
                 "tham chiếu thô": tc_tho, "tham chiếu đ.chỉnh": tc_dc,
                 "trần thô": tran_tho, "trần đ.chỉnh": tran_dc, "sàn thô": san_tho, "sàn đ.chỉnh": san_dc,
                 "lệch trần %": (tran_dc / tran_tho - 1) * 100})
tran_san_ls = pd.DataFrame(dong)
sai = tran_san_ls[(tran_san_ls["trần thô"] != tran_san_ls["trần đ.chỉnh"]) | (tran_san_ls["sàn thô"] != tran_san_ls["sàn đ.chỉnh"])]

print(f"{len(tran_san_ls)} lệnh trong 6 tháng · {len(sai)} lệnh ({len(sai) / max(len(tran_san_ls), 1):.0%}) "
      f"có trần/sàn sai nếu tính trên giá điều chỉnh, trên {sai['mã'].nunique()} mã")
if len(sai):
    print(f"Lệch trần lớn nhất {sai['lệch trần %'].abs().max():.2f}% · trung vị {sai['lệch trần %'].abs().median():.2f}%")
    duoi_san = sai[sai["trần đ.chỉnh"] < sai["sàn thô"]]
    print(f"{len(duoi_san)} lệnh có 'trần' điều chỉnh thấp hơn cả SÀN thật — LO đặt theo đó nằm ngoài biên độ: "
          f"{', '.join(dict.fromkeys(duoi_san['mã']))}")
sai.style.format({"phiên tín hiệu": "{:%d/%m/%Y}", "lệch trần %": "{:+.2f}%",
                  **{c: "{:,.0f}" for c in sai.columns if "thô" in c or "đ.chỉnh" in c}}).hide(axis="index")

60 lệnh trong 6 tháng · 15 lệnh (25%) có trần/sàn sai nếu tính trên giá điều chỉnh, trên 9 mã
Lệch trần lớn nhất 51.89% · trung vị 16.83%
9 lệnh có 'trần' điều chỉnh thấp hơn cả SÀN thật — LO đặt theo đó nằm ngoài biên độ: MSB, ACB, VJC, VHM, HDB


phiên tín hiệu,mã,chiều,tham chiếu thô,tham chiếu đ.chỉnh,trần thô,trần đ.chỉnh,sàn thô,sàn đ.chỉnh,lệch trần %
05/05/2026,HPG,BÁN,"27,300","24,370","29,200","26,050","25,400","22,700",-10.79%
05/05/2026,PVT,MUA,"23,750","21,590","25,400","23,100","22,100","20,100",-9.06%
07/05/2026,PVT,BÁN,"22,350","20,320","23,900","21,700","20,800","18,900",-9.21%
21/05/2026,NVL,BÁN,"15,300","14,130","16,350","15,100","14,250","13,150",-7.65%
21/05/2026,MSB,MUA,"14,450","12,040","15,450","12,850","13,450","11,200",-16.83%
01/06/2026,ACB,MUA,"24,900","21,450","26,600","22,950","23,200","19,950",-13.72%
09/06/2026,VJC,MUA,"179,000","137,690","191,500","147,300","166,500","128,100",-23.08%
12/06/2026,VHM,BÁN,"138,700","66,780","148,400","71,400","129,000","62,200",-51.89%
13/07/2026,HDB,MUA,"26,700","20,540","28,550","21,950","24,850","19,150",-23.12%
22/07/2026,HDB,BÁN,"25,800","19,850","27,600","21,200","24,000","18,500",-23.19%


In [18]:
# Cùng phép so trên mọi (mã, phiên) của rổ trong 6 tháng — không chỉ các phiên có lệnh
moi_phien = pd.DataFrame({"dc": dc, "tho": tho}).dropna()
ts = moi_phien.apply(lambda r: bt_vn.tran_san(r["dc"], SAN) != bt_vn.tran_san(r["tho"], SAN), axis=1)
o_T = moi_phien.xs(T, level="date")
sai_T = o_T[o_T["dc"] != o_T["tho"]]
print(f"Toàn rổ, 6 tháng: {ts.mean():.1%} số (mã, phiên) có trần/sàn sai nếu dùng giá điều chỉnh "
      f"({ts.groupby(level='symbol').any().sum()}/{len(VU_TRU)} mã bị ít nhất một phiên)")
print(f"Riêng phiên T = {T:%d/%m}: {len(sai_T)}/{len(o_T)} mã có giá điều chỉnh khác giá thô"
      + (f" — {', '.join(sai_T.index)}: GDKHQ ngay phiên kế tiếp" if len(sai_T) else ""))

Toàn rổ, 6 tháng: 40.6% số (mã, phiên) có trần/sàn sai nếu dùng giá điều chỉnh (23/30 mã bị ít nhất một phiên)


Riêng phiên T = 19/10: 0/30 mã có giá điều chỉnh khác giá thô


60 lệnh trong 6 tháng, **15 lệnh (25%)** sẽ in sai trần/sàn nếu tính trên giá
điều chỉnh, rải trên 9 mã. Trần lệch trung vị 16,83%, lớn nhất **51,89%** (VHM,
phiên 12/06). Mọi độ lệch đều âm: điều chỉnh lùi chỉ hạ giá quá khứ.

Mức lệch đó không phải sai số làm tròn. Ở 9 lệnh của 5 mã (MSB, ACB, VJC, VHM,
HDB), "giá trần" tính trên giá điều chỉnh còn **thấp hơn cả giá sàn thật** — một lệnh LO đặt theo bảng
đó bị Sở từ chối vì ngoài biên độ. Trên toàn rổ, 40,6% số cặp (mã, phiên) trong
6 tháng có trần/sàn sai, 23/30 mã dính ít nhất một phiên.

Dòng cuối mới là lý do lỗi này sống sót: ở **phiên T, 0/30 mã** có giá điều
chỉnh khác giá thô. Ai kiểm bằng cách "chạy hôm nay xem trần/sàn có khớp bảng
giá không" sẽ thấy khớp hoàn toàn — rồi mọi bảng lệnh dựng lại cho ngày đã qua,
kể cả phần đối chiếu nhật ký ở mục 12, âm thầm sai.

Một giới hạn còn lại: vào **chính ngày GDKHQ**, Sở điều chỉnh giá tham chiếu
theo quyền. finlens 1.4.0 chưa có lịch sự kiện quyền, nên bảng lệnh lập tối hôm
trước ngày GDKHQ in tham chiếu = đóng cửa thô, lệch với tham chiếu thật của phiên
mai. Lệnh ATO không mang giá nên không bị ảnh hưởng; cột trần/sàn thì có.

## 10 · Cạm bẫy (c): bảng lệnh hôm qua có đúng là lệnh backtest đã đặt hôm qua?

Đây là phép kiểm quan trọng nhất của một sản phẩm kiểu này. Bảng lệnh "phiên
mai" chỉ có nghĩa nếu nó **trùng khít** với lệnh mà backtest đặt ở cùng ngày khi
có thêm dữ liệu về sau. Cách kiểm:

1. Cắt dữ liệu tới phiên T−1 rồi chạy lại — đó là bảng lệnh notebook sẽ in nếu
   chạy tối hôm qua.
2. So với lệnh mà backtest đầy đủ (dữ liệu tới T) **thực sự đã tạo** ở thanh T−1.
3. Lặp lại cho 15 phiên gần nhất để có đủ ngày có lệnh.

Nếu hai bên lệch, backtest đang dùng một thứ mà đời thực không có lúc ra quyết
định. Để thấy phép kiểm bắt được lỗi, chạy song song một biến thể **nhìn trước**:
tính khối lượng mua theo `data.open[1]` — giá mở cửa ngày mai, chính là giá sẽ
khớp. Ý định rất tự nhiên (đỡ bị từ chối vì thiếu tiền khi giá mở cửa nhảy), và
backtrader **cho phép**: với `preload=True` cả mảng giá nằm sẵn trong bộ nhớ nên
chỉ số `[1]` đọc được tương lai mà không báo lỗi gì. Ở thanh cuối không có ngày
mai thì `IndexError` — và một dòng `try/except` lùi về giá đóng cửa là đủ để
che lỗi đi.

In [19]:
T_1 = phien_dong[-2]
s_T1 = chay(gia[gia["date"] <= T_1])
m_T1 = doi_chieu(lenh_cua(s, T_1), lenh_cua(s_T1, T_1))
print(f"Phiên T−1 = {T_1:%d/%m/%Y}: backtest đầy đủ đặt {int(m_T1['khoi_luong_dung'].notna().sum())} lệnh, "
      f"bảng lệnh cắt tới T−1 đề xuất {int(m_T1['khoi_luong_thu'].notna().sum())} lệnh, "
      f"{int((m_T1['ket_qua'] == 'trùng').sum())} trùng")
m_T1

Phiên T−1 = 16/10/2026: backtest đầy đủ đặt 0 lệnh, bảng lệnh cắt tới T−1 đề xuất 0 lệnh, 0 trùng


,ma,chieu,khoi_luong_dung,khoi_luong_thu,ket_qua


In [20]:
SO_PHIEN_LUI = 15
s_nt = chay(gia, nhin_truoc=True)

t0 = time.perf_counter()
dong = []
for bien_the, s_day_du in [("đúng", s), ("nhìn trước open[1]", s_nt)]:
    for c in phien_dong[-SO_PHIEN_LUI - 1:-1]:
        st = chay(gia[gia["date"] <= c], nhin_truoc=(s_day_du is s_nt))
        m = doi_chieu(lenh_cua(s_day_du, c), lenh_cua(st, c))
        for r in m.itertuples():
            dong.append({"biến thể": bien_the, "phiên": c, "mã": r.ma, "chiều": r.chieu,
                         "backtest đặt": r.khoi_luong_dung, "bảng lệnh đề xuất": r.khoi_luong_thu,
                         "kết quả": r.ket_qua})
nhat_quan = pd.DataFrame(dong)
print(f"{SO_PHIEN_LUI} phiên × 2 biến thể · {time.perf_counter() - t0:.0f} giây\n")

tong_hop = nhat_quan.groupby("biến thể").agg(
    phien_co_lenh=("phiên", "nunique"), so_lenh=("mã", "size"),
    trung=("kết quả", lambda k: int((k == "trùng").sum())),
)
tong_hop["lệch"] = tong_hop["so_lenh"] - tong_hop["trung"]
print(tong_hop.rename(columns={"phien_co_lenh": "phiên có lệnh", "so_lenh": "lệnh", "trung": "trùng"}).to_string())

15 phiên × 2 biến thể · 100 giây

                    phiên có lệnh  lệnh  trùng  lệch
biến thể                                            
nhìn trước open[1]              5    10      8     2
đúng                            5    10     10     0


In [21]:
nhat_quan[nhat_quan["kết quả"] != "trùng"].style.format(
    {"phiên": "{:%d/%m/%Y}", "backtest đặt": "{:,.0f}", "bảng lệnh đề xuất": "{:,.0f}"}, na_rep="—"
).hide(axis="index")

biến thể,phiên,mã,chiều,backtest đặt,bảng lệnh đề xuất,kết quả
nhìn trước open[1],30/09/2026,MSB,MUA,"25,700","25,800",lệch khối lượng
nhìn trước open[1],01/10/2026,PLX,MUA,"8,900","9,000",lệch khối lượng


In [22]:
tt_nt = bt_vn.ket_qua(s_nt)["tom_tat"]
print(f"Thành tích: đúng CAGR {tt['cagr']:+.2%}, {s.dem['bi_tu_choi']} lệnh bị từ chối · "
      f"nhìn trước CAGR {tt_nt['cagr']:+.2%}, {s_nt.dem['bi_tu_choi']} lệnh bị từ chối")

Thành tích: đúng CAGR +33.51%, 6 lệnh bị từ chối · nhìn trước CAGR +36.77%, 3 lệnh bị từ chối


Phiên T−1 (16/10) cả hai bên đều 0 lệnh — một phép so "trùng" chẳng nói lên
gì, nên mới cần lùi 15 phiên. Trên 15 phiên đó:

- **Bản đúng: 10/10 lệnh trùng khít** trên 5 phiên có lệnh — cùng mã, cùng
  chiều, cùng khối lượng. Bảng lệnh tối nay chính là thứ backtest sẽ ghi lại
  cho ngày hôm nay.
- **Bản nhìn trước: 2/10 lệnh lệch khối lượng** — MSB 30/09 (25.700 so với
  25.800) và PLX 01/10 (8.900 so với 9.000). Mỗi lần lệch đúng một lô.

Lệch một lô nhỏ tới mức dễ cho qua, và chính vì vậy nó nguy hiểm: nó là dấu vết
duy nhất cho thấy backtest đang ra quyết định bằng một giá chưa ai biết. Cái giá
của việc nhìn trước hiện ra ở dòng thành tích: CAGR **+36,77%** so với
**+33,51%**, số lệnh bị broker từ chối giảm từ 6 xuống 3 — backtest đẹp hơn
3,26 điểm phần trăm mỗi năm nhờ một thứ đời thực không có.

## 11 · Xuất file: HTML và Excel

Hai file, tên mang ngày của phiên tín hiệu T:

- **HTML** — một trang mở bằng trình duyệt: lời cảnh báo, các con số danh mục,
  bảng lệnh, danh mục đang giữ, đường vốn. plotly.js nhúng thẳng nên mở được
  không cần mạng.
- **Excel** — ba sheet `Lenh`, `Danh muc`, `Ghi chu`, số định dạng sẵn, dòng
  tiêu đề cố định.

In [23]:
CSS = """
<style>
  body { background:#f9f9f7; color:#0b0b0b; margin:0; padding:28px 32px 48px;
         font-family:system-ui,-apple-system,"Segoe UI",sans-serif; }
  h1 { font-size:22px; margin:0 0 2px; } h2 { font-size:16px; margin:26px 0 8px; }
  .moc { color:#898781; font-size:13px; margin-bottom:14px; }
  .canh-bao { background:#fff6e0; border:1px solid #eda100; border-radius:8px; padding:10px 14px;
              font-size:13px; margin-bottom:16px; }
  .bang-o { display:flex; gap:12px; flex-wrap:wrap; margin:8px 0 8px; }
  .o { flex:1 1 170px; background:#fcfcfb; border:1px solid rgba(11,11,11,.10); border-radius:10px; padding:12px 16px; }
  .nhan { font-size:12px; color:#52514e; } .gia-tri { font-size:22px; font-weight:600; margin-top:4px; }
  table { border-collapse:collapse; font-size:13px; background:#fcfcfb; }
  th, td { padding:6px 10px; border-bottom:1px solid #e1e0d9; text-align:right; }
  th { color:#52514e; font-weight:600; }
  td:first-child, th:first-child, td:last-child { text-align:left; }
  footer { color:#898781; font-size:12px; margin-top:24px; border-top:1px solid #e1e0d9; padding-top:12px; }
</style>
"""


def o(nhan, gia_tri):
    return f'<div class="o"><div class="nhan">{nhan}</div><div class="gia-tri">{gia_tri}</div></div>'


def bang_danh_muc_hien(dm: pd.DataFrame):
    if dm.empty:
        return pd.DataFrame({"": ["Không giữ mã nào"]}).style.hide(axis="index")
    return (
        dm.rename(columns={"ma": "Mã", "khoi_luong": "Khối lượng", "gia_von": "Giá vốn", "dong_cua_T": "Đóng cửa T",
                         "gia_tri": "Giá trị (VND)", "lai_lo_pct": "Lãi/lỗ", "ngay_mua": "Ngày khớp mua",
                         "ban_duoc_phien_toi": "Bán được phiên tới"})
        .style.format({"Khối lượng": "{:,.0f}", "Giá vốn": "{:,.0f}", "Đóng cửa T": "{:,.0f}",
                       "Giá trị (VND)": "{:,.0f}", "Lãi/lỗ": "{:+.2f}%", "Bán được phiên tới": "{:,.0f}",
                       "Ngày khớp mua": "{:%d/%m/%Y}"}, na_rep="—")
        .hide(axis="index")
    )


fig_von = duong(so_sanh, x="date", y="giá trị", theo="chuỗi", tieu_de="Đường vốn so với VNINDEX — gốc 100",
                nhan_y="chỉ số (gốc 100)", cao=380)

bang_lenh_html = (
    hien_bang_lenh(bang_lenh).to_html() if len(bang_lenh)
    else "<p>Không có lệnh cho phiên kế tiếp — giữ nguyên danh mục.</p>"
)
html = f"""<!doctype html>
<html lang="vi"><head><meta charset="utf-8"><title>Bảng lệnh sau phiên {T:%d/%m/%Y}</title>{CSS}</head>
<body>
  <h1>Bảng lệnh cho phiên kế tiếp — lập sau phiên {T:%d/%m/%Y}</h1>
  <div class="moc">Chạy lúc {GIO_SERVER:%H:%M %d/%m/%Y} (giờ server) · {len(VU_TRU)} mã HOSE thanh khoản cao ·
       đà tăng {PHIEN_DONG_LUONG} phiên + lọc SMA{PHIEN_MA} · tối đa {SO_VI_THE} vị thế</div>
  <div class="canh-bao">⚠️ Tài khoản giấy (paper trading). Sinh tự động từ dữ liệu FinLens; không kết nối broker,
       không đặt lệnh thật. <b>Không phải khuyến nghị đầu tư.</b> Giá tham chiếu/trần/sàn tính trên giá thô; giá trị
       lệnh theo giá tham chiếu — giá khớp ATO thật sẽ khác.</div>
  <div class="bang-o">{o("Giá trị danh mục giấy", f"{GIA_TRI_DM / 1e6:,.1f} tr")}{o("Tiền mặt", f"{TIEN_MAT / 1e6:,.1f} tr")}
       {o("Số vị thế", f"{len(danh_muc_T)} / {SO_VI_THE}")}{o("Số lệnh", str(len(bang_lenh)))}</div>
  <h2>Lệnh (loại ATO)</h2>{bang_lenh_html}
  <h2>Danh mục đang giữ sau phiên {T:%d/%m/%Y}</h2>{bang_danh_muc_hien(danh_muc_T).to_html()}
  <h2>Đường vốn</h2>{fig_von.to_html(full_html=False, include_plotlyjs=True)}
  <footer>Nguồn: FinLens · notebook <code>08-backtest-backtrader/89_bang_lenh_cuoi_ngay.ipynb</code></footer>
</body></html>"""

TEP_HTML = THU_MUC_RA / f"bang_lenh_{T:%Y%m%d}.html"
TEP_HTML.write_text(html, encoding="utf-8")
print(f"Đã ghi {TEP_HTML.name} ({TEP_HTML.stat().st_size / 1024:,.0f} KB)")

Đã ghi bang_lenh_20261019.html (4,789 KB)


In [24]:
from openpyxl.styles import Font
from openpyxl.utils import get_column_letter

ghi_chu = pd.DataFrame({"Mục": [
    "Phiên tín hiệu", "Áp dụng cho", "Chạy lúc (giờ server)", "Chiến lược", "Vũ trụ", "Vốn ban đầu (VND)",
    "Giá trị danh mục giấy (VND)", "Tiền mặt (VND)", "Giá trên lệnh", "Cảnh báo",
], "Nội dung": [
    f"{T:%d/%m/%Y}", "phiên giao dịch kế tiếp, lệnh ATO", f"{GIO_SERVER:%H:%M %d/%m/%Y}",
    f"đà tăng {PHIEN_DONG_LUONG} phiên, lọc SMA{PHIEN_MA}, tối đa {SO_VI_THE} vị thế, giữ khi còn top {NGUONG_GIU}",
    f"{len(VU_TRU)} mã HOSE GTGD cao nhất 3 tháng: {', '.join(VU_TRU)}", VON_BAN_DAU, round(GIA_TRI_DM), round(TIEN_MAT),
    "giá thô (adjusted=False), VND; trần/sàn ±7% theo bước giá HOSE",
    "Paper trading. Không kết nối broker, không đặt lệnh thật. Không phải khuyến nghị đầu tư.",
]})

TEP_XLSX = THU_MUC_RA / f"bang_lenh_{T:%Y%m%d}.xlsx"
with pd.ExcelWriter(TEP_XLSX, engine="openpyxl") as w:
    bang_lenh.drop(columns="ngay_tin_hieu").rename(columns=TEN_COT).to_excel(w, sheet_name="Lenh", index=False)
    danh_muc_T.to_excel(w, sheet_name="Danh muc", index=False)
    ghi_chu.to_excel(w, sheet_name="Ghi chu", index=False)
    for ten, ws in w.sheets.items():
        ws.freeze_panes = "A2"
        for i, cot in enumerate(ws.iter_cols(min_row=1, max_row=ws.max_row), start=1):
            rong = max(len(str(c.value)) if c.value is not None else 0 for c in cot)
            ws.column_dimensions[get_column_letter(i)].width = min(max(rong + 2, 10), 80)
            for c in cot[1:]:
                if isinstance(c.value, (int, float)):
                    c.number_format = "#,##0.00" if ten == "Danh muc" and cot[0].value == "lai_lo_pct" else "#,##0"
        for c in ws[1]:
            c.font = Font(bold=True)

print(f"Đã ghi {TEP_XLSX.name}: Lenh {len(bang_lenh)} dòng · Danh muc {len(danh_muc_T)} dòng · Ghi chu")

Đã ghi bang_lenh_20261019.xlsx: Lenh 0 dòng · Danh muc 5 dòng · Ghi chu


## 12 · Nhật ký paper-trading

Mỗi lần chạy, bảng lệnh của phiên T được **nối thêm** vào
`output/nhat_ky_paper_trading.csv` — một dòng mỗi lệnh, hoặc một dòng
`KHÔNG LỆNH` nếu hôm đó không có lệnh, để phân biệt "đã chạy, không có gì" với
"quên chạy". **Phiên T đã có trong nhật ký thì không ghi lại**: chạy notebook
hai lần trong một tối không nhân đôi lệnh, và một lần chạy lại sau này không
viết đè lên những gì đã đề xuất thật.

CSV thay vì parquet vì người dùng mở nó bằng Excel; ngày lưu dạng ISO.

In [25]:
COT_NHAT_KY = ["ngay_tin_hieu", "ma", "chieu", "khoi_luong", "loai_lenh", "gia_tham_chieu", "gia_tran", "gia_san",
               "gia_tri_lenh", "ly_do", "chay_luc"]


def doc_nhat_ky(tep: Path) -> pd.DataFrame:
    if not tep.exists():
        return pd.DataFrame(columns=COT_NHAT_KY)
    return pd.read_csv(tep, parse_dates=["ngay_tin_hieu"], dtype={"ma": "string", "chieu": "string"})


nhat_ky = doc_nhat_ky(NHAT_KY)
da_co = set(pd.to_datetime(nhat_ky["ngay_tin_hieu"]))

if T in da_co:
    print(f"Phiên {T:%d/%m/%Y} đã có trong nhật ký ({(nhat_ky['ngay_tin_hieu'] == T).sum()} dòng) — không ghi thêm.")
else:
    moi = (
        bang_lenh.drop(columns="buoc_gia") if len(bang_lenh)
        else pd.DataFrame([{"ngay_tin_hieu": T, "chieu": "KHÔNG LỆNH"}])
    ).assign(chay_luc=GIO_SERVER.isoformat(timespec="seconds")).reindex(columns=COT_NHAT_KY)
    nhat_ky = moi if nhat_ky.empty else pd.concat([nhat_ky, moi], ignore_index=True)
    nhat_ky.to_csv(NHAT_KY, index=False, date_format="%Y-%m-%d")
    print(f"Đã nối {len(moi)} dòng của phiên {T:%d/%m/%Y} vào {NHAT_KY.name}")

nhat_ky = doc_nhat_ky(NHAT_KY)
print(f"Nhật ký: {len(nhat_ky)} dòng · {nhat_ky['ngay_tin_hieu'].nunique()} phiên "
      f"({nhat_ky['ngay_tin_hieu'].min():%d/%m/%Y} → {nhat_ky['ngay_tin_hieu'].max():%d/%m/%Y})")

Phiên 19/10/2026 đã có trong nhật ký (1 dòng) — không ghi thêm.
Nhật ký: 1 dòng · 1 phiên (19/10/2026 → 19/10/2026)


### Đối chiếu: lệnh đã đề xuất với giá khớp thật

Lệnh ATO khớp ở **giá mở cửa thô của phiên sau** phiên tín hiệu. Khi phiên đó
đã đóng cửa và có trong dữ liệu, mỗi dòng nhật ký nhận được giá mở cửa thật,
độ trượt so với giá tham chiếu đã in, và câu trả lời cho "lệnh có khớp được
không" — mở cửa trần (lệnh mua) hoặc sàn (lệnh bán) mà cả phiên một giá là
không khớp, đúng luật `KhopTheoBienDo` của backtest.

Cột cuối so giá khớp thật với giá **backtest đã khớp** cho cùng lệnh — phép kiểm
mục 10 làm trên dữ liệu đời thực.

In [26]:
def doi_chieu_gia_khop(nk: pd.DataFrame, gia_tho: pd.DataFrame, giao_dich_bt: pd.DataFrame) -> pd.DataFrame:
    """Gắn giá mở cửa thô của phiên sau vào từng lệnh nhật ký, nếu phiên đó đã có."""
    nk = nk[nk["chieu"].isin(["MUA", "BÁN"])]
    phien = np.array(sorted(gia_tho["date"].unique()))
    bang = gia_tho.set_index(["symbol", "date"])
    bt_gia = giao_dich_bt.assign(chieu=np.where(giao_dich_bt["so_cp"] > 0, "MUA", "BÁN")).set_index(
        ["symbol", "date", "chieu"])["gia"]
    ra = []
    for l in nk.itertuples():
        sau = phien[phien > np.datetime64(l.ngay_tin_hieu)]
        if len(sau) == 0 or (l.ma, sau[0]) not in bang.index:
            continue
        p = pd.Timestamp(sau[0])
        th = bang.loc[(l.ma, p)]
        mo = th["open"] * VND_MOI_DON_VI
        trang_ben = th["high"] == th["low"] and (
            (l.chieu == "MUA" and mo >= l.gia_tran) or (l.chieu == "BÁN" and mo <= l.gia_san))
        gia_bt = bt_gia.get((l.ma, p, l.chieu), np.nan)
        ra.append({"phiên tín hiệu": l.ngay_tin_hieu, "phiên khớp": p, "mã": l.ma, "chiều": l.chieu,
                   "khối lượng": l.khoi_luong, "tham chiếu": l.gia_tham_chieu, "mở cửa thật": mo,
                   "trượt %": (mo / l.gia_tham_chieu - 1) * 100,
                   "khớp được": "không — trắng bên" if trang_ben else "có",
                   "backtest khớp": gia_bt, "thật − backtest": mo - gia_bt})
    return pd.DataFrame(ra)


DINH_DANG_KHOP = {"phiên tín hiệu": "{:%d/%m/%Y}", "phiên khớp": "{:%d/%m/%Y}", "khối lượng": "{:,.0f}",
                  "tham chiếu": "{:,.0f}", "mở cửa thật": "{:,.0f}", "trượt %": "{:+.2f}%",
                  "backtest khớp": "{:,.0f}", "thật − backtest": "{:+,.0f}"}

khop_that = doi_chieu_gia_khop(nhat_ky, gia_tho, kq["giao_dich"])
cho = nhat_ky[nhat_ky["chieu"].isin(["MUA", "BÁN"])]
print(f"Nhật ký có {len(cho)} lệnh · {len(khop_that)} lệnh đã có giá mở cửa của phiên sau để đối chiếu")
if khop_that.empty:
    print("Chưa đối chiếu được lệnh nào — chạy lại notebook sau khi phiên kế tiếp đóng cửa.")
else:
    display(khop_that.style.format(DINH_DANG_KHOP, na_rep="—").hide(axis="index"))

Nhật ký có 0 lệnh · 0 lệnh đã có giá mở cửa của phiên sau để đối chiếu
Chưa đối chiếu được lệnh nào — chạy lại notebook sau khi phiên kế tiếp đóng cửa.


Trong lúc nhật ký còn mỏng, cùng hàm đối chiếu áp lên **lệnh backtest của 20
phiên gần nhất**, lập lại bằng `lap_bang_lenh` như thể notebook đã chạy mỗi tối.
Đây không phải nhật ký — chỉ để biết khoảng trượt giữa giá tham chiếu in trên
lệnh và giá ATO thật thường cỡ nào.

In [27]:
ls20 = pd.DataFrame(s.so_lenh, columns=COT_LENH)
ls20 = ls20[ls20["ngay"].isin(phien_dong[-21:-1])]
gia_lap = doi_chieu_gia_khop(lap_bang_lenh(ls20, gia_tho), gia_tho, kq["giao_dich"])

print(f"{len(gia_lap)} lệnh · trượt mở cửa so với tham chiếu: trung vị |{gia_lap['trượt %'].abs().median():.2f}|% · "
      f"lớn nhất {gia_lap['trượt %'].abs().max():.2f}% · không khớp được {(gia_lap['khớp được'] != 'có').sum()}")
print(f"Giá thật − giá backtest khớp: lệch tuyệt đối lớn nhất {gia_lap['thật − backtest'].abs().max():,.0f} VND/cp")
gia_lap.style.format(DINH_DANG_KHOP, na_rep="—").hide(axis="index")

10 lệnh · trượt mở cửa so với tham chiếu: trung vị |0.32|% · lớn nhất 0.95% · không khớp được 0
Giá thật − giá backtest khớp: lệch tuyệt đối lớn nhất 0 VND/cp


phiên tín hiệu,phiên khớp,mã,chiều,khối lượng,tham chiếu,mở cửa thật,trượt %,khớp được,backtest khớp,thật − backtest
30/09/2026,01/10/2026,STB,BÁN,"4,700","74,000","74,100",+0.14%,có,"74,100",+0
30/09/2026,01/10/2026,MSB,MUA,"24,700","14,200","14,300",+0.70%,có,"14,300",+0
01/10/2026,02/10/2026,VIC,BÁN,"1,400","220,000","218,500",-0.68%,có,"218,500",+0
01/10/2026,02/10/2026,PLX,MUA,"8,300","36,650","37,000",+0.95%,có,"37,000",+0
05/10/2026,06/10/2026,PLX,BÁN,"8,300","35,500","35,650",+0.42%,có,"35,650",+0
05/10/2026,06/10/2026,VIC,MUA,"1,200","232,000","232,000",+0.00%,có,"232,000",+0
09/10/2026,12/10/2026,VIC,BÁN,"1,200","225,500","225,000",-0.22%,có,"225,000",+0
09/10/2026,12/10/2026,VPB,MUA,"11,900","23,800","23,800",+0.00%,có,"23,800",+0
15/10/2026,16/10/2026,BSR,BÁN,"12,100","28,200","28,400",+0.71%,có,"28,400",+0
15/10/2026,16/10/2026,TCB,MUA,"10,300","32,900","32,900",+0.00%,có,"32,900",+0


Trên 10 lệnh của 20 phiên gần nhất: giá mở cửa thật lệch giá tham chiếu in trên
lệnh trung vị **0,32%**, lớn nhất 0,95% (PLX 01/10); không lệnh nào rơi vào phiên
trắng bên. Cột cuối bằng 0 ở mọi dòng: giá backtest khớp trùng giá mở cửa thô
tới từng đồng, vì trong 20 phiên này các mã liên quan không có GDKHQ nên giá
điều chỉnh bằng giá thô.

Độ trượt cỡ 0,3% là lý do khối lượng mua chỉ dùng 98% phần vốn của mỗi vị thế:
giá trị lệnh in theo tham chiếu, còn tiền thật trừ theo giá ATO.

## Tổng kết

| Bước | Chi tiết dễ bỏ sót | Đo được ở lần chạy này |
|---|---|---|
| Phiên dở dang | `bo_phien_chua_dong` theo đồng hồ server; vũ trụ lọc tới **hôm qua** | chạy lúc 11:30: sai **2/2** phiên có lệnh, 0/4 lệnh đúng còn lại |
| Giá trên lệnh | tham chiếu, trần, sàn trên **giá thô** — giá điều chỉnh chỉ đúng ở phiên mới nhất | **15/60** lệnh (25%) sai trần/sàn, lệch tới 51,89%; riêng phiên T: 0/30 mã |
| Nhất quán | bảng lệnh = `so_lenh` của thanh cuối; kiểm bằng cách chạy lùi từng phiên | đúng **10/10** trùng; nhìn trước 8/10, CAGR đẹp hơn 3,26 đpt |
| Đơn vị | đọc `attrs` một lần trước mọi `concat`, truyền `don_vi_gia=` tường minh | frame ghép ở mục 8 vẫn chạy đúng |
| Nhật ký | nối thêm, khoá theo phiên tín hiệu, không ghi đè | chạy lại không nhân đôi |

### Chạy hằng ngày

Chạy sau 15:00 mỗi phiên (cần khoá API đã cấu hình):

```bash
venv\Scripts\jupyter.exe nbconvert --to notebook --execute --inplace notebooks/08-backtest-backtrader/89_bang_lenh_cuoi_ngay.ipynb
```

Ba file nằm trong `output/`: `bang_lenh_YYYYMMDD.html`, `bang_lenh_YYYYMMDD.xlsx`
và `nhat_ky_paper_trading.csv` dày thêm mỗi ngày. Sau vài tuần, phần đối chiếu
ở mục 12 mới bắt đầu có ý nghĩa thống kê.

**Và điều cuối:** notebook dừng ở file. Từ file tới lệnh thật là một bước người
dùng tự làm, tự chịu trách nhiệm — và một chiến lược chưa qua walk-forward
([`87_toi_uu_va_walk_forward.ipynb`](87_toi_uu_va_walk_forward.ipynb)) và chưa
có vài tháng nhật ký giấy thì chưa nên đi bước đó.

---

**Track 8 hết.** Quay lại đầu track:
[`81_tu_finlens_toi_cerebro.ipynb`](81_tu_finlens_toi_cerebro.ipynb) · đối chiếu
với backtest vectorised: [`34_backtest_chien_luoc.ipynb`](../03-phan-tich-ky-thuat/34_backtest_chien_luoc.ipynb)